In [3]:
import os

path = r"C:\Users\Admin\Downloads\infosys\data\raw\patients .csv"
print(os.path.exists(path))

True


In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\patients .csv")
df

,Patient_ID,Name,Age,Gender,Diagnosis
0,P00001,Patient_1,82,Female,Fever
1,P00002,Patient_2,31,Male,Diabetes
2,P00003,Patient_3,15,Male,Fever
3,P00004,Patient_4,50,Female,Asthma
4,P00005,Patient_5,17,Female,Diabetes
...,...,...,...,...,...
4995,P04996,Patient_4996,50,Male,Cancer
4996,P04997,Patient_4997,67,Female,Diabetes
4997,P04998,Patient_4998,33,Male,Asthma
4998,P04999,Patient_4999,90,Female,Diabetes


In [7]:
print(f"info: {df.info}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


info: <bound method DataFrame.info of      Patient_ID          Name  Age  Gender Diagnosis
0        P00001     Patient_1   82  Female     Fever
1        P00002     Patient_2   31    Male  Diabetes
2        P00003     Patient_3   15    Male     Fever
3        P00004     Patient_4   50  Female    Asthma
4        P00005     Patient_5   17  Female  Diabetes
...         ...           ...  ...     ...       ...
4995     P04996  Patient_4996   50    Male    Cancer
4996     P04997  Patient_4997   67  Female  Diabetes
4997     P04998  Patient_4998   33    Male    Asthma
4998     P04999  Patient_4999   90  Female  Diabetes
4999     P05000  Patient_5000   73  Female    Cancer

[5000 rows x 5 columns]>
Shape: (5000, 5)
Columns: ['Patient_ID', 'Name', 'Age', 'Gender', 'Diagnosis']


In [8]:
print(df.head())

  Patient_ID       Name  Age  Gender Diagnosis
0     P00001  Patient_1   82  Female     Fever
1     P00002  Patient_2   31    Male  Diabetes
2     P00003  Patient_3   15    Male     Fever
3     P00004  Patient_4   50  Female    Asthma
4     P00005  Patient_5   17  Female  Diabetes


In [9]:
print(f"\nDtypes:\n{df.dtypes}")

print(f"\nNull counts:\n{df.isna().sum()}")


Dtypes:
Patient_ID    object
Name          object
Age            int64
Gender        object
Diagnosis     object
dtype: object

Null counts:
Patient_ID    0
Name          0
Age           0
Gender        0
Diagnosis     0
dtype: int64


In [10]:
print(f"\nDuplicate rows: {df.duplicated().sum()}")

for col in df.select_dtypes(include='object').columns:
    print(f"\nUnique values in '{col}': {sorted(df[col].astype(str).unique())[:20]}")


Duplicate rows: 0

Unique values in 'Patient_ID': ['P00001', 'P00002', 'P00003', 'P00004', 'P00005', 'P00006', 'P00007', 'P00008', 'P00009', 'P00010', 'P00011', 'P00012', 'P00013', 'P00014', 'P00015', 'P00016', 'P00017', 'P00018', 'P00019', 'P00020']

Unique values in 'Name': ['Patient_1', 'Patient_10', 'Patient_100', 'Patient_1000', 'Patient_1001', 'Patient_1002', 'Patient_1003', 'Patient_1004', 'Patient_1005', 'Patient_1006', 'Patient_1007', 'Patient_1008', 'Patient_1009', 'Patient_101', 'Patient_1010', 'Patient_1011', 'Patient_1012', 'Patient_1013', 'Patient_1014', 'Patient_1015']

Unique values in 'Gender': ['Female', 'Male']

Unique values in 'Diagnosis': ['Asthma', 'Cancer', 'Diabetes', 'Fever']


In [11]:
df.nunique()

Patient_ID    5000
Name          5000
Age             90
Gender           2
Diagnosis        4
dtype: int64

In [15]:
print(f"Age range: {df['Age'].min()} - {df['Age'].max()}")


Age range: 1 - 90


In [18]:
print("STANDARDIZE GENDER")

GENDER_MAP = {
    'm': 'Male', 'male': 'Male',
    'f': 'Female', 'female': 'Female', 'femal': 'Female',
    'o': 'Other', 'other': 'Other', 'others': 'Other'
}

before = sorted(df['Gender'].astype(str).unique())
df['Gender'] = (
    df['Gender'].astype(str).str.strip().str.lower()
      .map(GENDER_MAP)
      .fillna(df['Gender'].astype(str).str.strip().str.title())
)
after = sorted(df['Gender'].unique())

print(f"Before: {before}")
print(f"After : {after}")
assert set(df['Gender'].unique()) <= {'Male', 'Female', 'Other'}, \
    "Gender contains values outside Male/Female/Other"
print("PASSED: Gender restricted to Male / Female / Other")

STANDARDIZE GENDER
Before: ['Female', 'Male']
After : ['Female', 'Male']
PASSED: Gender restricted to Male / Female / Other


In [19]:
print("VALIDATE AGE")
 
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
invalid_age = df['Age'].isna().sum()
if invalid_age > 0:
    print(f"WARNING: {invalid_age} non-numeric Age value(s) coerced to NaN")
 
print(f"Age range: {df['Age'].min()} - {df['Age'].max()}")
 

VALIDATE AGE
Age range: 1 - 90


In [20]:
print("STANDARDIZE DIAGNOSIS")
 
before_diag = sorted(df['Diagnosis'].astype(str).unique())
df['Diagnosis'] = (
    df['Diagnosis'].astype(str).str.strip()
      .str.replace(r'\s+', ' ', regex=True)
      .str.title()
)
after_diag = sorted(df['Diagnosis'].unique())
print(f"Before: {before_diag}")
print(f"After : {after_diag}")

STANDARDIZE DIAGNOSIS
Before: ['Asthma', 'Cancer', 'Diabetes', 'Fever']
After : ['Asthma', 'Cancer', 'Diabetes', 'Fever']


In [23]:
print("FLAG CLINICAL ANOMALIES")
 
anomaly_mask = (df['Age'] > 110) | (df['Age'] < 0) | (df['Age'].isna())
df['Anomaly_Flag'] = anomaly_mask
print(f"Rows flagged as anomalies: {int(anomaly_mask.sum())}")
print(df['Anomaly_Flag'].value_counts())
 

FLAG CLINICAL ANOMALIES
Rows flagged as anomalies: 0
Anomaly_Flag
False    5000
Name: count, dtype: int64


In [27]:
admissions = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\admissions.csv")
print(admissions.head())



  Admission_ID Patient_ID Doctor_ID Department_ID Admission_Date  \
0       A00001     P00001   DR00186          D002     2025-10-20   
1       A00002     P00002   DR00281          D019     2025-01-21   
2       A00003     P00003   DR00105          D020     2025-06-24   
3       A00004     P00004   DR00312          D004     2025-07-23   
4       A00005     P00005   DR00430          D018     2025-04-06   

  Discharge_Date           Status  
0     2025-10-28         Critical  
1     2025-01-23        Recovered  
2     2025-07-03       Discharged  
3     2025-07-25         Critical  
4     2025-04-07  Under Treatment  


In [28]:
import pandas as pd

patients = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\patients .csv")
admissions = pd.read_csv(r"C:\Users\Admin\Downloads\infosys\data\raw\admissions.csv")

# Check if all Patient_IDs in admissions exist in patients
missing_ids = admissions[~admissions["Patient_ID"].isin(patients["Patient_ID"])]

if missing_ids.empty:
    print("All Patient_IDs in admissions.csv exist in patients.csv")
else:
    print("Some Patient_IDs are missing in patients.csv")
    print("Count of missing IDs:", missing_ids["Patient_ID"].nunique())
    print("Example missing IDs:", missing_ids["Patient_ID"].unique()[:10])


All Patient_IDs in admissions.csv exist in patients.csv


In [33]:
print("FINAL VALIDATION & SAVE")
 
final_cols = ['Patient_ID', 'Gender', 'Age', 'Diagnosis', 'Anomaly_Flag']
df = df[final_cols]
 
checks = {
    "No nulls remaining":            df.isna().sum().sum() == 0,
    "No duplicate rows":             df.duplicated().sum() == 0,
    "Patient_ID is unique":          df['Patient_ID'].is_unique,
    "Gender in {Male,Female,Other}": set(df['Gender'].unique()) <= {'Male', 'Female', 'Other'}
   
}
for check, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")
 
assert all(checks.values()), "Validation failed — see checks above."
print(f"\nFinal shape: {df.shape}")
print(df.head(5).to_string(index=False))


FINAL VALIDATION & SAVE
  [PASS] No nulls remaining
  [PASS] No duplicate rows
  [PASS] Patient_ID is unique
  [PASS] Gender in {Male,Female,Other}

Final shape: (5000, 5)
Patient_ID Gender  Age Diagnosis  Anomaly_Flag
    P00001 Female   82     Fever         False
    P00002   Male   31  Diabetes         False
    P00003   Male   15     Fever         False
    P00004 Female   50    Asthma         False
    P00005 Female   17  Diabetes         False


In [ ]:
 
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved cleaned file to: {C:\Users\Admin\Downloads}"

In [36]:
df.to_csv("patients_clean.csv", index=False)

In [ ]:
#df.to_csv(OUTPUT_PATH, index=False)
#print(f"\nSaved cleaned file to: {C:\Users\Admin\Downloads\infosys}")

In [35]:
df.to_csv("patients_clean.csv", index=False)
df

,Patient_ID,Gender,Age,Diagnosis,Anomaly_Flag
0,P00001,Female,82,Fever,False
1,P00002,Male,31,Diabetes,False
2,P00003,Male,15,Fever,False
3,P00004,Female,50,Asthma,False
4,P00005,Female,17,Diabetes,False
...,...,...,...,...,...
4995,P04996,Male,50,Cancer,False
4996,P04997,Female,67,Diabetes,False
4997,P04998,Male,33,Asthma,False
4998,P04999,Female,90,Diabetes,False
